# 02 · Feature engineering

All features are computed by `src.features.compute_features` from adjusted OHLCV. Each feature at day *t* uses
rows ≤ *t* only. They are scale-invariant (returns, ratios, oscillators) because Yahoo's adjusted *price levels* are
back-adjusted with future corporate actions.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from src.utils import load_config
cfg = load_config(ROOT / "config.yaml")
RESULTS = ROOT / cfg["experiment"]["results_dir"]

In [ ]:
from src.data_loader import load_ticker
from src.features import compute_features, feature_table, feature_groups
ticker = cfg["data"]["tickers"][0]
prices, meta = load_ticker(ticker, cfg["data"]["start"], cfg["data"]["end"], ROOT / cfg["data"]["raw_dir"])
F = compute_features(prices, cfg["features"])
print(ticker, F.shape)
feature_table()

## Warm-up periods
Indicators need a history window; warm-up rows are NaN and are dropped (never imputed).

In [ ]:
F.isna().sum().sort_values(ascending=False).head(10).to_frame("NaN rows (warm-up)")

## Point-in-time check (truncation invariance)
If features used future data, recomputing them on a truncated history would change past values.

In [ ]:
cuts = [600, 1500, 2500, 3500]
ok = all(np.allclose(compute_features(prices.iloc[:c], cfg["features"]).iloc[-1].values, F.iloc[c - 1].values, equal_nan=True) for c in cuts)
print("Features at t identical when all data after t is removed:", ok)

## Feature correlation (training period)
Strongly correlated groups (e.g. SMA/EMA ratios) are expected. Tree models cope with this, and Logistic Regression is regularised.

In [ ]:
train = F.loc[: cfg["split"]["train_end"]].dropna()
corr = train.corr()
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=7); ax.set_yticks(range(len(corr)), corr.columns, fontsize=7)
fig.colorbar(im, shrink=0.7); ax.set_title(f"{ticker}: feature correlation, training period"); plt.show()

## Feature relevance
Mutual information is measured on the training data and permutation importance on the validation data. Importance shows what a model relies on. It does **not** establish causality.

In [ ]:
imp_dir = RESULTS / "metrics" / "feature_importance"
mi = pd.read_csv(imp_dir / f"{ticker}_mutual_information.csv")
perm = pd.read_csv(imp_dir / f"{ticker}_xgboost_permutation.csv")
mi.head(15).merge(perm, on="feature", how="left").round(5)

In [ ]:
for name in [f"{ticker}_xgboost_permutation_importance.png", f"{ticker}_technical_indicators.png"]:
    p = RESULTS / "figures" / name
    if p.exists(): display(Image(filename=str(p)))

## Feature-group ablation (XGBoost, test period)

In [ ]:
pd.read_csv(RESULTS / "metrics" / "feature_ablation.csv").round(4)